# Hyperparameter Tuning

This notebook focuses on optimizing the performance of machine learning models by tuning their hyperparameters. Techniques such as Grid Search or Randomized Search are used to identify the best parameter combinations, improving model accuracy and generalization on unseen data.

In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score,confusion_matrix

df = pd.read_csv('../data/processed/clean_data.csv')
print("Data Loaded Successfully")

Data Loaded Successfully


## Splitting Features and Target Variable

The dataset is divided into **features (`X`)** and the **target variable (`y`)**. The `label` column is assigned as the target, while all remaining columns are used as input features for model training.

In [2]:
X = df.drop('label',axis=1)
y = df['label']

## Splitting the Dataset into Training and Testing Sets

The dataset is divided into training and testing sets using an **80:20 ratio**. The training set is used to build the machine learning model, while the testing set is used to evaluate its performance on unseen data.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

## Random Forest Hyperparameter Tuning using GridSearchCV
GridSearchCV is used to find the best Random Forest parameters by testing multiple combinations. The optimized model is evaluated using training score, test accuracy, and confusion matrix.

In [4]:
model = RandomForestClassifier(random_state=42)

# Hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

print("\n\nRandomForestClf Hyperparameter Tuning ....")

# Apply GridSearchCV
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Train model with best parameters
# grid_search.fit(X_train, y_train)

# # Best model
# best_model = grid_search.best_estimator_

# # Predictions
# y_pred = best_model.predict(X_test)

# print(f"Best Parameters => {grid_search.best_params_}")
# print(f"Training Score => {best_model.score(X_train, y_train)}")
# print(f"Test Accuracy => {accuracy_score(y_test, y_pred)}")
# print(f"Confusion Matrix => \n{confusion_matrix(y_test, y_pred)}")



RandomForestClf Hyperparameter Tuning ....


## Hyperparameter Tuning of the CatBoost Classifier

The `CatBoostClassifier` is optimized using **GridSearchCV**, which evaluates multiple combinations of hyperparameters through 5-fold cross-validation. After identifying the best parameter set, the optimized model is trained and evaluated using the training score, cross-validation score, test accuracy, and confusion matrix to measure its classification performance.

In [5]:
cat_model = CatBoostClassifier(verbose=0, random_state=48)

# Hyperparameter grid
param_grid = {
    'iterations': [100, 200],
    'learning_rate': [0.01, 0.1],
    'depth': [4, 6, 8],
    'l2_leaf_reg': [1, 3, 5]
}

# Grid Search
grid_search = GridSearchCV(
    estimator=cat_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

print("\n\nCatBoost Hyperparameter Tuning...")
grid_search.fit(X_train, y_train)

# Best model
best_model = grid_search.best_estimator_

# Predictions
y_pred = best_model.predict(X_test)

# Results
print("\nBest Parameters:", grid_search.best_params_)
print(f"Best Cross Validation Score => {grid_search.best_score_:.4f}")
print(f"Training Score => {best_model.score(X_train, y_train):.4f}")
print(f"Test Accuracy => {accuracy_score(y_test, y_pred):.4f}")
print(f"Confusion Matrix =>\n{confusion_matrix(y_test, y_pred)}")



CatBoost Hyperparameter Tuning...
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best Parameters: {'depth': 4, 'iterations': 100, 'l2_leaf_reg': 1, 'learning_rate': 0.01}
Best Cross Validation Score => 1.0000
Training Score => 1.0000
Test Accuracy => 1.0000
Confusion Matrix =>
[[   28     0]
 [    0 22516]]


## Hyperparameter Tuning of the LightGBM Classifier

The `LGBMClassifier` is optimized using **GridSearchCV**, which evaluates different combinations of hyperparameters through 5-fold cross-validation. The best-performing model is then selected and evaluated using the cross-validation score, training score, test accuracy, and confusion matrix to measure its classification performance.

In [6]:
lgbm_model = LGBMClassifier(random_state=48)

# Hyperparameter grid
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [5, 10, -1],
    'num_leaves': [31, 50, 70]
}

# Grid Search
grid_search = GridSearchCV(
    estimator=lgbm_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

print("\n\nLGBM Hyperparameter Tuning...")
grid_search.fit(X_train, y_train)

# Best model
best_model = grid_search.best_estimator_

#save mode
joblib.dump(best_model, "C:/Office 2019/DS_PW/ML_Practice/Customer_Categorizer/phishing-classification/models/best_model.pkl")

# Predictions
y_pred = best_model.predict(X_test)

# Results
print("\nBest Parameters:", grid_search.best_params_)
print(f"Best Cross Validation Score => {grid_search.best_score_:.4f}")
print(f"Training Score => {best_model.score(X_train, y_train):.4f}")
print(f"Test Accuracy => {accuracy_score(y_test, y_pred):.4f}")
print(f"Confusion Matrix =>\n{confusion_matrix(y_test, y_pred)}")



LGBM Hyperparameter Tuning...
Fitting 5 folds for each of 36 candidates, totalling 180 fits
[LightGBM] [Info] Number of positive: 90054, number of negative: 118
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001399 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1007
[LightGBM] [Info] Number of data points in the train set: 90172, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.998691 -> initscore=6.637480
[LightGBM] [Info] Start training from score 6.637480
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive

## Hyperparameter Tuning of the XGBoost Classifier

The `XGBClassifier` is optimized using **GridSearchCV**, which systematically searches for the best combination of hyperparameters through 5-fold cross-validation. The optimized model is then evaluated using the best cross-validation score, training score, test accuracy, and confusion matrix to assess its overall classification performance.

In [7]:
xgb_model = XGBClassifier(
    random_state=48,
    eval_metric='logloss',
    use_label_encoder=False
)

# Hyperparameter grid
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Grid Search
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

print("\n\nXGBoost Hyperparameter Tuning...")
grid_search.fit(X_train, y_train)

# Best model
best_model = grid_search.best_estimator_

# Predictions
y_pred = best_model.predict(X_test)

# Results
print("\nBest Parameters:", grid_search.best_params_)
print(f"Best Cross Validation Score => {grid_search.best_score_:.4f}")
print(f"Training Score => {best_model.score(X_train, y_train):.4f}")
print(f"Test Accuracy => {accuracy_score(y_test, y_pred):.4f}")
print(f"Confusion Matrix =>\n{confusion_matrix(y_test, y_pred)}")



XGBoost Hyperparameter Tuning...
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}
Best Cross Validation Score => 1.0000
Training Score => 1.0000
Test Accuracy => 1.0000
Confusion Matrix =>
[[   28     0]
 [    0 22516]]


C:\Users\ajeet\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\training.py:199: UserWarning: [01:53:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


## Save Test Dataset

The test features and target variable are combined into a single DataFrame and saved as a CSV file. This exported dataset can be used later for model testing, deployment, or external validation.

In [8]:
pd.concat([X_test,y_test],axis=1).to_csv('../data/external/test.csv',index=False)